Below are tests for the various heuristics in our paper.  
Change the parameters `d` and `p` in the first cell below to parametrize the tests.  
It is necessary that $p = 3 \mod 4$ for the tests to run properly.  
The claims made in the paper assume that the quadratic field $K = \mathbb{Q}(\sqrt{d})$ has trivial narrow class group.

In [1]:
d = 5
p = (2**110).next_prime()

In [2]:
from rm_data import standard_RM
from nf_utility import *
from represent_integer import RepresentIntegerSolver

from itertools import product

In [3]:
x = polygen(QQ)
K.<aK> = NumberField(x^2 - d)
B.<i,j,k> = QuaternionAlgebra(QQ, -1, -p)
O0 = B.maximal_order(order_basis=(B.one(), i, 1/2 * (i + j), 1/2 * (1 + k)))
a = standard_RM(K, O0)

## Heuristic 7.1
We check that we may find a small element in `ideal` with prime norm.

In [4]:
ideal = a.O_BK.random_ideal()
basis = ideal.reduced_integral_basis()
_, N = is_ideal_narrow_principal(ideal.norm())
q_values = [sum(c*e for c, e in zip(coeffs, basis)).reduced_norm() / N for coeffs in product(range(-1, 2), repeat=8)]
any(q.is_prime() and K.ideal(q).residue_symbol(2, 2) != 1 for q in q_values)

True

## Heuristic 7.2
We check that a random ideal class produces a balanced quadratic form.  
Because the heuristic statement is asymptotic, we perform the test with various values for $p$.

In [5]:
p_1 = (2^32).next_prime()
p_2 = (2^65).next_prime()
p_3 = (2**110).next_prime()

def balance_defect(p, n_tests=10):
    B.<i,j,k> = QuaternionAlgebra(QQ, -1, -p)
    O0 = B.maximal_order(order_basis=(B.one(), i, 1/2 * (i + j), 1/2 * (1 + k)))
    a = standard_RM(K, O0)
    res = 0
    for _ in range(n_tests):
        ideal = a.O_BK.random_ideal()
        basis = ideal.reduced_integral_basis()
        def q(x):
            return x.reduced_norm().trace()
        balance_defect = max(q(x) for x in basis) / min(q(x) for x in basis)
        res += balance_defect / n_tests
    return res

print(f"For a 32 bits prime, we get an expected balance defect of {balance_defect(p_1).n(16)}")
print(f"For a 65 bits prime, we get an expected balance defect of {balance_defect(p_2).n(16)}")
print(f"For a 110 bits prime, we get an expected balance defect of {balance_defect(p_3).n(16)}")

For a 32 bits prime, we get an expected balance defect of 1.913
For a 65 bits prime, we get an expected balance defect of 1.787
For a 110 bits prime, we get an expected balance defect of 1.583


## Heuristic 7.4
We check that the expectation of $\ell^{e_0} = max(1, p(log(p))/\sqrt{N_{K/\mathbb{Q}}(N_J)})$ is asymptotically close to $\sqrt{p}/\sqrt{d}$, for $N_J$ the output of the `equivalent_ideal_prime_norm` with input a random quaternion ideal.

In [6]:
p_1 = (2^32).next_prime()
p_2 = (2^65).next_prime()
p_3 = (2**110).next_prime()

def average(p, n_tests=10):
    B.<i,j,k> = QuaternionAlgebra(QQ, -1, -p)
    O0 = B.maximal_order(order_basis=(B.one(), i, 1/2 * (i + j), 1/2 * (1 + k)))
    a = standard_RM(K, O0)
    res = 0
    for _ in range(n_tests):
        ideal = a.O_BK.random_ideal()
        J, _, _ = ideal.equivalent_ideal_prime_norm([2])
        value = max(1, p * log(p)/sqrt(J.norm().norm()))
        res += value / n_tests
    return res

print(f"We find an expected value of {average(p_1).n(16)} for ell^e_0, against a prediction of {(sqrt(p_1) / sqrt(d)).n(16)}.")
print(f"We find an expected value of {average(p_2).n(16)} for ell^e_0, against a prediction of {(sqrt(p_2) / sqrt(d)).n(16)}.")
print(f"We find an expected value of {average(p_3).n(16)} for ell^e_0, against a prediction of {(sqrt(p_3) / sqrt(d)).n(16)}.")

We find an expected value of 443300. for ell^e_0, against a prediction of 29310..
We find an expected value of 8.062e10 for ell^e_0, against a prediction of 2.716e9.
We find an expected value of 8.895e17 for ell^e_0, against a prediction of 1.611e16.


## Heuristic 7.5
We check that field elements output by the sample rejection algorithm are prime at the expected frequency.
We also check that among the prime outputs, the residue symbol of $-1$ with respect to them also behaves randomly.

In [10]:
solver = RepresentIntegerSolver(K, p)
R = K.ring_of_integers()

_, M_1 = is_ideal_narrow_principal(K.ideal(R.random_element(ceil(2 * p * log(p)))))
_, M_2 = is_ideal_narrow_principal(K.ideal(R.random_element(p**2)))
_, M_3 = is_ideal_narrow_principal(K.ideal(R.random_element(p**3)))
    
def run_test(M, n_rounds=1000):
    prob_prime = 0
    prob_residue = 0
    sampler = solver.sample_Z(M)
    for i in range(n_rounds):
        z = next(sampler)
        z.__repr__() #Weirdly enough, the computation raises an error without this useless line.
        if z.is_prime():
            prob_prime += ~n_rounds
            if K.ideal(z).residue_symbol(-1, 2) == 1:
                prob_residue += ~n_rounds
    prob_residue /= prob_prime
    print(f"We found a probability {prob_prime.n(8)} of getting a prime output, against a heuristic probability of {~log(M.norm()).n(8)}.")
    print(f"The probability that a prime outpus satisfies the residue condition was {prob_residue.n(8)} against a heuristic 0.5.")

run_test(M_1)
run_test(M_2)
run_test(M_3)

We found a probability 0.26 of getting a prime output, against a heuristic probability of 0.0061.
The probability that a prime outpus satisfies the residue condition was 0.52 against a heuristic 0.5.
We found a probability 0.024 of getting a prime output, against a heuristic probability of 0.0033.
The probability that a prime outpus satisfies the residue condition was 0.50 against a heuristic 0.5.
We found a probability 0.011 of getting a prime output, against a heuristic probability of 0.0022.
The probability that a prime outpus satisfies the residue condition was 0.46 against a heuristic 0.5.


## Heuristic 7.7
We check that for $M > cp^2$, the probability of $M - ps^2 - pt^2$ being prime behaves like for elements of norm bounded by $N_{K/\mathbb{Q}}(M/p)$.  
We also check that among the prime outputs, the residue symbol of $-1$ also behaves randomly.  
#### Warning:
This function takes a while to run, this is to be expected.

In [11]:
solver = RepresentIntegerSolver(K, p)
R = K.ring_of_integers()
_, M = is_ideal_narrow_principal(K.ideal(p^2 * R.random_element(2^10)+ R.random_element(floor(p/2))))

def run_test(M, n_rounds=1000):
    prob_prime = 0
    prob_residue = 0
    sampler = solver.sample_Z(M)
    for _ in range(n_rounds):
        z = 1
        done = False
        while not done:
            z = next(sampler)
            if z.is_prime():
                done, zw = solver.solve_two_squares(z)
        z, w = zw
        val = M - p*z**2 - p*w**2
        if val.is_prime():
            prob_prime += ~n_rounds
            if K.ideal(val).residue_symbol(-1, 2) == 1:
                prob_residue += ~n_rounds
    prob_residue /= prob_prime
    print(f"We found a probability {prob_prime.n(8)} of getting a prime output, against a heuristic probability of {~log((M/p).norm()).n(8)}.")
    print(f"The probability that a prime output satisfies the residue condition was {prob_residue.n(8)} against a heuristic 0.5.")

run_test(M)

We found a probability 0.0020 of getting a prime output, against a heuristic probability of 0.0060.
The probability that a prime output satisfies the residue condition was 0.50 against a heuristic 0.5.
